# Paper — 00: Re-run SAM2 with fixed NMS threshold

**Why:** `postprocess_match_threshold` was changed from 0.2 → 0.5 in commit `ceb7b84`,
matching that of the original experiments. YOLO kept 0.2. Because SAM2 masks are
larger than YOLO boxes, they have higher pairwise IoU → NMS at 0.5 drops far more
SAM2 detections (~10k) than YOLO detections (~26k). Re-running with 0.2 restores
apples-to-apples comparison.

**Fix:** `postprocess_match_threshold=0.2` for all SAM2 variants.

**Outputs:** Three new dirs in `~/tmp/YOLOv8BeyondEarth/`:
- `exp_sam2_fixed_256`
- `exp_sam2_ft_fixed_256`
- `exp_sam2_auto_fixed_256`

**Requires GPU** (`--gres=gpu:1` on Sherlock). YOLO does not need re-running.

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import torch
import torch._utils
if not hasattr(torch, "_utils"):
    torch._utils = sys.modules["torch._utils"]

import time
from pathlib import Path

from sahi import AutoDetectionModel
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

from YOLOv8BeyondEarth.SAM2_predict import get_sliced_prediction_SAM2, get_sliced_prediction_SAM2_auto

In [ ]:
torch.cuda.empty_cache()
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. On Sherlock: srun --gres=gpu:1 --mem=32G --pty bash")
print(f"GPU: {torch.cuda.get_device_name(0)}")

work_dir  = Path.home() / "tmp" / "YOLOv8BeyondEarth"
in_raster = Path("/scratch/users/cayleigh/test_raster/M1221383405.tif")

model_weights   = work_dir / "yolov8_model" / "yolov8-m-boulder-detection-tmp.pt"
sam2_checkpoint = Path("/scratch/users/cayleigh/checkpoints/sam2.1_hiera_small.pt")
sam2_ft_weights = Path("/scratch/users/cayleigh/sam2_finetuned/sam2_boulder_best.pt")

sam2_fixed_dir      = work_dir / "exp_sam2_fixed_256"
sam2_ft_fixed_dir   = work_dir / "exp_sam2_ft_fixed_256"
sam2_auto_fixed_dir = work_dir / "exp_sam2_auto_fixed_256"
for d in [sam2_fixed_dir, sam2_ft_fixed_dir, sam2_auto_fixed_dir]:
    d.mkdir(parents=True, exist_ok=True)

SLICE_SIZE           = 256
INFERENCE_SIZE       = 1024
OVERLAP              = 0.2
CONFIDENCE_THRESHOLD = 0.10
MIN_AREA_THRESHOLD   = 6
NMS_THRESHOLD        = 0.5

print(f"NMS threshold: {NMS_THRESHOLD}")

In [ ]:
torch.cuda.empty_cache()

detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=model_weights.as_posix(),
    confidence_threshold=CONFIDENCE_THRESHOLD,
    device="cuda:0",
    image_size=INFERENCE_SIZE)

# SAM2 zero-shot
sam2_base = build_sam2("configs/sam2.1/sam2.1_hiera_s.yaml", sam2_checkpoint, device="cuda:0")
predictor = SAM2ImagePredictor(sam2_base)

# SAM2 fine-tuned
sam2_ft_model = build_sam2("configs/sam2.1/sam2.1_hiera_s.yaml", sam2_checkpoint, device="cuda:0")
sam2_ft_model.load_state_dict(torch.load(sam2_ft_weights))
predictor_ft = SAM2ImagePredictor(sam2_ft_model)

# SAM2 auto
mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2_base,
    points_per_side=32,
    pred_iou_thresh=0.50,
    stability_score_thresh=0.85,
    min_mask_region_area=MIN_AREA_THRESHOLD)

print("Models loaded.")

In [ ]:
# SAM2 zero-shot — skip if already run
existing = list(sam2_fixed_dir.glob("*-downscaled-mask-nms.shp"))
if existing:
    print(f"SAM2 zero-shot already exists ({len(existing)} file(s)), skipping.")
else:
    print("Running SAM2 zero-shot (NMS=0.2)...")
    t0 = time.time()
    get_sliced_prediction_SAM2(
        in_raster,
        predictor,
        detection_model=detection_model,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        output_dir=sam2_fixed_dir,
        slice_size=SLICE_SIZE,
        inference_size=INFERENCE_SIZE,
        overlap_height_ratio=OVERLAP,
        overlap_width_ratio=OVERLAP,
        min_area_threshold=MIN_AREA_THRESHOLD,
        downscale_pred=True,
        postprocess=True,
        postprocess_match_threshold=NMS_THRESHOLD,
        postprocess_class_agnostic=False,
        batch_size=16)
    print(f"Done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# SAM2 fine-tuned — skip if already run
existing = list(sam2_ft_fixed_dir.glob("*-downscaled-mask-nms.shp"))
if existing:
    print(f"SAM2 fine-tuned already exists ({len(existing)} file(s)), skipping.")
else:
    print("Running SAM2 fine-tuned (NMS=0.2)...")
    t0 = time.time()
    get_sliced_prediction_SAM2(
        in_raster,
        predictor_ft,
        detection_model=detection_model,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        output_dir=sam2_ft_fixed_dir,
        slice_size=SLICE_SIZE,
        inference_size=INFERENCE_SIZE,
        overlap_height_ratio=OVERLAP,
        overlap_width_ratio=OVERLAP,
        min_area_threshold=MIN_AREA_THRESHOLD,
        downscale_pred=True,
        postprocess=True,
        postprocess_match_threshold=NMS_THRESHOLD,
        postprocess_class_agnostic=False,
        batch_size=16)
    print(f"Done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# SAM2 auto — skip if already run
existing = list(sam2_auto_fixed_dir.glob("*-mask-nms.shp"))
if existing:
    print(f"SAM2 auto already exists ({len(existing)} file(s)), skipping.")
else:
    print("Running SAM2 auto (NMS=0.2)...")
    t0 = time.time()
    get_sliced_prediction_SAM2_auto(
        in_raster,
        mask_generator=mask_generator,
        output_dir=sam2_auto_fixed_dir,
        slice_size=SLICE_SIZE,
        overlap_height_ratio=OVERLAP,
        overlap_width_ratio=OVERLAP,
        min_area_threshold=MIN_AREA_THRESHOLD,
        postprocess=True,
        postprocess_match_threshold=NMS_THRESHOLD)
    print(f"Done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# Quick sanity check — compare counts to pre-fix runs
import geopandas as gpd

checks = [
    ("YOLO (unchanged)",           work_dir / "exp_yolo_256",          "*-downscaled-mask-nms.shp"),
    ("SAM2 zero-shot (pre-fix)",   work_dir / "exp_sam2_256",          "*-downscaled-mask-nms.shp"),
    ("SAM2 zero-shot (fixed)",     sam2_fixed_dir,                      "*-downscaled-mask-nms.shp"),
    ("SAM2 fine-tuned (pre-fix)",  work_dir / "exp_sam2_finetuned_256", "*-downscaled-mask-nms.shp"),
    ("SAM2 fine-tuned (fixed)",    sam2_ft_fixed_dir,                   "*-downscaled-mask-nms.shp"),
    ("SAM2 auto (pre-fix)",        work_dir / "exp_sam2_auto_256",      "*-mask-nms.shp"),
    ("SAM2 auto (fixed)",          sam2_auto_fixed_dir,                 "*-mask-nms.shp"),
]

print(f"{'Model':<35}  {'detections':>12}")
print("-" * 50)
for label, d, glob in checks:
    shps = list(d.glob(glob))
    if not shps:
        print(f"{label:<35}  {'(no shp)':>12}")
    else:
        n = sum(len(gpd.read_file(p)) for p in shps)
        print(f"{label:<35}  {n:>12}")